In [1]:
import pandas as pd
from sklearn.model_selection import KFold
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import balanced_accuracy_score
import warnings

warnings.filterwarnings('ignore')

file_path = '../../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

catboost_model = CatBoostClassifier(random_state=42, silent=True)
lr_model = LogisticRegression(random_state=42)
rf_model = RandomForestClassifier(random_state=42)
et_model = ExtraTreesClassifier(random_state=42)
gb_model = GradientBoostingClassifier(n_estimators=100, random_state=42)

bagging_lr = BaggingClassifier(lr_model, random_state=42)
bagging_rf = BaggingClassifier(rf_model, random_state=42)
bagging_et = BaggingClassifier(et_model, random_state=42)
bagging_gb = BaggingClassifier(gb_model, random_state=42)

catboost_preds = []
lr_preds = []
rf_preds = []
et_preds = []
gb_preds = []
meta_labels = []

for train_idx, val_idx in kf.split(X):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    catboost_model.fit(X_train, y_train)
    bagging_lr.fit(X_train, y_train)
    bagging_rf.fit(X_train, y_train)
    bagging_et.fit(X_train, y_train)
    bagging_gb.fit(X_train, y_train)

    catboost_preds.append(catboost_model.predict(X_val))
    lr_preds.append(bagging_lr.predict(X_val))
    rf_preds.append(bagging_rf.predict(X_val))
    et_preds.append(bagging_et.predict(X_val))
    gb_preds.append(bagging_gb.predict(X_val))

    meta_labels.append(y_val)

X_meta = pd.DataFrame({
    'catboost': [item for sublist in catboost_preds for item in sublist],
    'lr': [item for sublist in lr_preds for item in sublist],
    'rf': [item for sublist in rf_preds for item in sublist],
    'et': [item for sublist in et_preds for item in sublist],
    'gb': [item for sublist in gb_preds for item in sublist]
})

In [2]:
X_meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8637 entries, 0 to 8636
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   catboost  8637 non-null   object
 1   lr        8637 non-null   int64 
 2   rf        8637 non-null   int64 
 3   et        8637 non-null   int64 
 4   gb        8637 non-null   int64 
dtypes: int64(4), object(1)
memory usage: 337.5+ KB


In [3]:
X_meta.head()

,catboost,lr,rf,et,gb
0,[0],0,0,0,0
1,[2],2,2,2,2
2,[0],0,0,0,0
3,[0],0,0,0,0
4,[1],1,1,1,1


In [4]:
# CatBoost возвращает предсказания в формате [число] (список с одним элементом), поэтому нужно убрать скобки, извлекая сами числа.

X_meta_copy = X_meta.copy()
X_meta_copy['catboost'] = X_meta_copy['catboost'].apply(lambda x: x[0])

X_meta_copy.head()

,catboost,lr,rf,et,gb
0,0,0,0,0,0
1,2,2,2,2,2
2,0,0,0,0,0
3,0,0,0,0,0
4,1,1,1,1,1


In [5]:
X_meta_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8637 entries, 0 to 8636
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   catboost  8637 non-null   int64
 1   lr        8637 non-null   int64
 2   rf        8637 non-null   int64
 3   et        8637 non-null   int64
 4   gb        8637 non-null   int64
dtypes: int64(5)
memory usage: 337.5 KB


In [6]:
X_meta_copy = X_meta_copy.reset_index(drop=True)

In [7]:
y_meta = [item for sublist in meta_labels for item in sublist]

meta_model = LogisticRegression(random_state=42)

meta_model.fit(X_meta_copy, y_meta)

catboost_model.fit(X, y)
bagging_lr.fit(X, y)
bagging_rf.fit(X, y)
bagging_et.fit(X, y)
bagging_gb.fit(X, y)

catboost_final_preds = catboost_model.predict(X)
lr_final_preds = bagging_lr.predict(X)
rf_final_preds = bagging_rf.predict(X)
et_final_preds = bagging_et.predict(X)
gb_final_preds = bagging_gb.predict(X)

X_meta_test = pd.DataFrame({
    'catboost': catboost_final_preds.ravel(),
    'lr': lr_final_preds,
    'rf': rf_final_preds,
    'et': et_final_preds,
    'gb': gb_final_preds
})

final_preds = meta_model.predict(X_meta_test)

accuracy = balanced_accuracy_score(y, final_preds)
print(f'Accuracy: {accuracy:.4f}')

Accuracy: 0.9987
